In [0]:
spark.sql("SHOW TABLES IN parts.bronze").show(truncate=False);

In [0]:
%sql
Select * From parts.bronze.autoloader_logs

In [0]:
import requests, json
from databricks import dbutils

key = dbutils.secrets.get("openai","api_key")
endpoint = "https://oai-embeddings.openai.azure.com"
deployment = "test-embedding-3-small"
api_version = "2024-02-15-preview"

url = f"{endpoint}/openai/deployments/{deployment}/embeddings?api-version={api_version}"
resp = requests.post(url, headers={"api-key": key, "Content-Type":"application/json"},
                     data=json.dumps({"input":["test"]}))
print(resp.status_code, resp.text)

In [0]:
key = dbutils.secrets.get("openai","api_key")
print("key_length:", len(key))
print("key_prefix:", key[:4])

In [0]:
import requests, json
endpoint = "https://oai-embeddings.openai.azure.com"
deployment = "text-embedding-3-small"
api_version = "2024-02-15-preview"

url = f"{endpoint}/openai/deployments/{deployment}/embeddings?api-version={api_version}"
resp = requests.post(url, headers={"api-key": key, "Content-Type": "application/json"},
                     data=json.dumps({"input":["test"]}))
print(resp.status_code, resp.text)

In [0]:
%sql
-- Genera métricas por tabla
SELECT
  table_name,
  total_rows,
  embedded_rows,
  ROUND(embedded_rows * 100.0 / NULLIF(total_rows, 0), 2) AS embedded_pct
FROM (
  SELECT
    'screws_bolts' AS table_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN embedding IS NOT NULL THEN 1 ELSE 0 END) AS embedded_rows
  FROM parts.bronze.screws_bolts
  UNION ALL
  SELECT
    'gaskets' AS table_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN embedding IS NOT NULL THEN 1 ELSE 0 END) AS embedded_rows
  FROM parts.bronze.gaskets
  UNION ALL
  SELECT
    'bushings' AS table_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN embedding IS NOT NULL THEN 1 ELSE 0 END) AS embedded_rows
  FROM parts.bronze.bushings
) t
ORDER BY table_name;